In [1]:
import pandas as pd

from data_loader import TwinsDataLoader, LalondeDataLoader, ACSDataLoader, IHDPDataLoader, WalmartDataLoader
from utils import apply_data_preparations_seq
from utils import bin_equal_frequency_2, bin_equal_frequency_5, bin_equal_frequency_10, bin_equal_width_2, \
    bin_equal_width_5, bin_equal_width_10, min_max_norm, log_norm, zscore_clip_3, zscore_filter_3, winsorize_aux, IQR, \
    isolationForest
from utils import calculate_ate_linear_regression_lstsq, calculate_ate_with_uncertainty
from experiments import largest_data_transformations


loaded cached data
loaded cached data
loaded cached data
loaded cached data


In [83]:
ate_bins_data_ihdp = pd.read_csv("ate_bins_data_ihdp.csv")
ate_bins_data_acs = pd.read_csv("ate_bins_data_acs.csv")
ate_bins_data_lalonde = pd.read_csv("ate_bins_data_lalonde.csv")
ate_bins_data_twins = pd.read_csv("ate_bins_data_twins.csv")

In [3]:
# largest_data_transformations = {
#     "bin_equal_frequency_2": bin_equal_frequency_2,
#     "bin_equal_frequency_5": bin_equal_frequency_5,
#     "bin_equal_frequency_10": bin_equal_frequency_10,
#     "bin_equal_width_2": bin_equal_width_2,
#     "bin_equal_width_5": bin_equal_width_5,
#     "bin_equal_width_10": bin_equal_width_10,
#     "norm_min_max": min_max_norm,
#     "norm_log": log_norm,
#     "zscore_clip_3": zscore_clip_3,
#     "zscore_filter_3": zscore_filter_3,
#     "winsorize": winsorize_aux,
#     "IQR": IQR,
#     "isolationForest": isolationForest
# }

df_twins = TwinsDataLoader().load_data().dropna()
df_lalonde = LalondeDataLoader().load_data().dropna()
df_acs = ACSDataLoader().load_data().dropna()
df_IHDP = IHDPDataLoader().load_data().dropna()
df_walmart = WalmartDataLoader().load_data()

loaded cached data
loaded cached data
loaded cached data
loaded cached data


In [9]:
print(f"walmart has dupes? {df_walmart.duplicated().sum()}")
print(f"twins has dupes? {df_twins.duplicated().sum()}")
print(f"lalonde has dupes? {df_lalonde.duplicated().sum()}")
print(f"acs has dupes? {df_acs.duplicated().sum()}")
print(f"ihdp has dupes? {df_IHDP.duplicated().sum()}")

print(len(df_acs), len(df_acs.drop_duplicates()))

walmart has dupes? 3
twins has dupes? 0
lalonde has dupes? 27
acs has dupes? 841363
ihdp has dupes? 0
1188308 346945


In [3]:
common_causes_twins = df_twins.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_lalonde = df_lalonde.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_acs = df_acs.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_ihdp = df_IHDP.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_walmart = df_walmart.columns.difference(["treatment", "outcome"], sort=False).tolist()

In [89]:
def get_n_percentile(df, n, start_ate, to_the_left=False):
    # 2. Calculate cumulative sum of counts
    df_ = df.copy()
    if to_the_left:
        df_ = df_.iloc[::-1].copy()
    if to_the_left:
        df_filtered = df_[df_['min_ate_in_bin'] <= start_ate].copy()
    else:
        df_filtered = df_[df_['min_ate_in_bin'] >= start_ate].copy()
    df_filtered['cum_sum'] = df_filtered['count'].cumsum()

    # 3. Find the 75th percentile target value
    total_elements = df_filtered['count'].sum()
    target = total_elements * (n / 100)

    # 4. Get the first bucket where cumulative sum is >= target
    p_bucket = df_filtered[df_filtered['cum_sum'] >= target].iloc[0]['bucket_range']
    print(f"{n}th Percentile Bucket (starting from {start_ate}){" to the right" if not to_the_left else "to the left"}: {p_bucket}")

    return p_bucket

In [93]:
# get_n_percentile(ate_bins_data_twins, 75, 0.06)
get_n_percentile(ate_bins_data_twins, 75, 0.06, True)
get_n_percentile(ate_bins_data_twins, 85, 0.06, True)
get_n_percentile(ate_bins_data_twins, 90, 0.06, True)
get_n_percentile(ate_bins_data_twins, 95, 0.06, True)
print("*"*20)
# get_n_percentile(ate_bins_data_lalonde, 75, 1671)
get_n_percentile(ate_bins_data_lalonde, 75, 1671, True)
print("*"*20)
get_n_percentile(ate_bins_data_acs, 75, 8774)
# get_n_percentile(ate_bins_data_acs, 75, 8774, True)
print("*"*20)
get_n_percentile(ate_bins_data_ihdp, 75, 3.92)
# get_n_percentile(ate_bins_data_ihdp, 75, 3.92, True)

75th Percentile Bucket (starting from 0.06)to the left: (0.036897, 0.036902]
85th Percentile Bucket (starting from 0.06)to the left: (0.013404, 0.013408]
90th Percentile Bucket (starting from 0.06)to the left: (0.0019709, 0.0019757]
95th Percentile Bucket (starting from 0.06)to the left: (0.0019325, 0.0019373]
********************
90th Percentile Bucket (starting from 1671)to the left: (1300.568, 1305.269]
********************
75th Percentile Bucket (starting from 8774) to the right: (14478.876, 14639.036]
********************
75th Percentile Bucket (starting from 3.92) to the right: (3.9379, 3.9382]


'(3.9379, 3.9382]'

In [86]:


def parse_bucket(bucket_str):
    """
    Parses bucket strings like:
    '(3.9379, 3.9382]'

    Returns:
        (bucket_min, bucket_max)
    """

    numbers = re.findall(r"-?\d+(?:\.\d+)?", bucket_str)

    if len(numbers) != 2:
        raise ValueError(f"Could not parse bucket: {bucket_str}")

    return float(numbers[0]), float(numbers[1])


def get_min_length_for_bucket_str(df, bucket_str, epsilon):
    """
    Returns the minimum `min_length` among all buckets
    intersecting with the given bucket string.
    """

    bucket_min, bucket_max = parse_bucket(bucket_str)

    mask = (
        (df["max_ate_in_bin"] >= bucket_min - epsilon) &
        (df["min_ate_in_bin"] <= bucket_max + epsilon)
    )

    matching_rows = df[mask]
    if matching_rows.empty:
        return None  # or np.nan

    return matching_rows["min_length"].min()

In [122]:
for p in range(50,100,5):
    start_ate = 0.06
    bucket = get_n_percentile(ate_bins_data_twins, p, start_ate, True)
    bucket_min, bucket_max = parse_bucket(bucket)
    print(f"TWINS, p: {p}")
    print(f"left, min is:{get_min_length_for_bucket_str(ate_bins_data_twins, bucket,0.012)}")
    print(f"right, min is:{get_min_length_for_bucket_str(ate_bins_data_twins, str([2*start_ate - bucket_max, 2*start_ate - bucket_min]),0.012)}")

    start_ate = 8774
    bucket = get_n_percentile(ate_bins_data_acs, p, 8774)
    bucket_min, bucket_max = parse_bucket(bucket)
    print(f"ACS, p: {p}")
    print(f"left, min is: {get_min_length_for_bucket_str(ate_bins_data_acs, str([2*start_ate - bucket_max, 2*start_ate - bucket_min]),286.5)}")
    print(f"right, min is: {get_min_length_for_bucket_str(ate_bins_data_acs, bucket,286.5)}")
    print("~"*100)

50th Percentile Bucket (starting from 0.06)to the left: (0.04953, 0.049535]
TWINS, p: 50
left, min is:0
right, min is:0
50th Percentile Bucket (starting from 8774) to the right: (11275.657, 11435.818]
ACS, p: 50
left, min is: 1
right, min is: 1
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
55th Percentile Bucket (starting from 0.06)to the left: (0.047192, 0.047197]
TWINS, p: 55
left, min is:1
right, min is:0
55th Percentile Bucket (starting from 8774) to the right: (11275.657, 11435.818]
ACS, p: 55
left, min is: 1
right, min is: 1
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
60th Percentile Bucket (starting from 0.06)to the left: (0.046761, 0.046765]
TWINS, p: 60
left, min is:1
right, min is:0
60th Percentile Bucket (starting from 8774) to the right: (13357.749, 13517.91]
ACS, p: 60
left, min is: None
right, min is: 2
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [115]:
a = ate_bins_data_twins[ate_bins_data_twins['min_ate_in_bin'] < 0.061]['count'].sum()
b = ate_bins_data_twins[((ate_bins_data_twins['min_ate_in_bin'] < 0.061) & (ate_bins_data_twins['min_ate_in_bin'] > 0.06 ))]['count'].sum()

print(f"a: {a}, b: {b}, b is {100 * b /a }% of a")

a: 18848, b: 8453, b is 44.848259762309% of a


In [7]:
def print_results_bucket(df, common_causes, ate_bins_data):
    cant = False
    transformed_df = apply_data_preparations_seq(df.copy(), ast.literal_eval(ate_bins_data.iloc[0]['best_sequence']), largest_data_transformations)
    pre_ate = calculate_ate_linear_regression_lstsq(df.copy(), 'treatment', 'outcome',common_causes)
    new_ate = calculate_ate_linear_regression_lstsq(transformed_df.copy(), 'treatment', 'outcome',common_causes)
    uncertainty_start = calculate_ate_with_uncertainty(df.copy(), 'treatment', 'outcome', common_causes)
    try:
        uncertainty_end = calculate_ate_with_uncertainty(transformed_df.copy(), 'treatment', 'outcome', common_causes)
    except:
        cant = True

    print(f"ATE went from: {pre_ate} to {new_ate} ({((new_ate - pre_ate) / pre_ate) * 100}% change)")
    print(F"Uncertainty start: {uncertainty_start['ci']}")
    if not cant:
        print(F"Uncertainty end: {uncertainty_end['ci']}")
    print(f"length of sequcene: {ate_bins_data.iloc[0]['min_length']}")

In [8]:
print("----- TWINS -----")
print_results_bucket(df_twins, common_causes_twins, ate_bins_data_twins)
print("\n----- LALONDE -----")
print_results_bucket(df_lalonde, common_causes_lalonde, ate_bins_data_lalonde)
print("\n----- ACS -----")
print_results_bucket(df_acs, common_causes_acs, ate_bins_data_acs)
print("\n----- IHDP -----")
print_results_bucket(df_IHDP, common_causes_ihdp, ate_bins_data_ihdp)

----- TWINS -----
ATE went from: 0.06132348739538075 to -0.00886169841806177 (-114.45074113434926% change)
Uncertainty start: (np.float64(0.049479750924671696), np.float64(0.0731672238660898))
Uncertainty end: (np.float64(-0.02226529439315853), np.float64(0.004541897557034991))
length of sequcene: 3

----- LALONDE -----
ATE went from: 1671.1304157875102 to -29.78031122296866 (-101.78204590986006% change)
Uncertainty start: (np.float64(417.25402062059015), np.float64(2925.00681095443))
length of sequcene: 5

----- ACS -----
ATE went from: 8774.433205040823 to 5670.0238825786655 (-35.380169293199536% change)
Uncertainty start: (np.float64(8487.97525927715), np.float64(9060.891150804497))
Uncertainty end: (np.float64(5385.387870712644), np.float64(5954.659894444687))
length of sequcene: 3

----- IHDP -----
ATE went from: 3.9286717508727045 to 3.8894029577622127 (-0.9995437542413346% change)
Uncertainty start: (np.float64(3.7071774141869174), np.float64(4.150166087558492))
Uncertainty end:

In [90]:
(ate_bins_data_lalonde.head(20))
# ate_bins_data_acs.iloc[0]['best_sequence']

,bucket_range,min_ate_in_bin,max_ate_in_bin,min_length,best_sequence,count
0,"(-29.78, -25.079]",-29.780311,-29.780311,5,"(('zscore_clip_3', 'age'), ('bin_equal_frequen...",1
1,"(346.29, 350.991]",350.882028,350.882028,5,"(('norm_log', 'age'), ('zscore_clip_3', 'age')...",1
2,"(383.897, 388.598]",388.334061,388.334061,5,"(('zscore_clip_3', 'age'), ('bin_equal_frequen...",1
3,"(388.598, 393.299]",390.078195,390.078195,4,"(('bin_equal_frequency_10', 'age'), ('zscore_f...",8
4,"(393.299, 398.0]",393.525670,393.525670,5,"(('bin_equal_frequency_10', 'age'), ('zscore_f...",1
5,"(398.0, 402.701]",400.665787,401.174070,4,"(('zscore_clip_3', 'age'), ('bin_equal_frequen...",8
6,"(407.401, 412.102]",407.426267,407.426267,5,"(('zscore_filter_3', 'age'), ('bin_equal_frequ...",1
7,"(477.915, 482.615]",482.606617,482.606617,5,"(('bin_equal_frequency_10', 'age'), ('zscore_c...",1
8,"(487.316, 492.017]",491.081198,491.081198,5,"(('zscore_filter_3', 'age'), ('bin_equal_frequ...",1
9,"(492.017, 496.718]",494.804015,495.635888,5,"(('bin_equal_frequency_2', 'age'), ('zscore_cl...",2


Index(['bucket_range', 'min_ate_in_bin', 'max_ate_in_bin', 'min_length',
       'best_sequence', 'count'],
      dtype='object')

In [9]:
print(ate_bins_data_twins.iloc[0]['best_sequence'])
print("-"*50)
print(ate_bins_data_acs.iloc[0]['best_sequence'])
print("-"*50)
print(ate_bins_data_lalonde.iloc[0]['best_sequence'])
print("-"*50)
print(ate_bins_data_ihdp.iloc[0]['best_sequence'])

(('bin_equal_width_2', 'adequacy'), ('norm_log', 'gestat10'), ('bin_equal_frequency_2', 'wt'))
--------------------------------------------------
(('bin_equal_width_2', 'Age'), ('bin_equal_frequency_5', 'education'), ('norm_log', 'insurance through employer'))
--------------------------------------------------
(('bin_equal_width_5', 'age'), ('zscore_clip_3', 'age'), ('norm_log', 'education'), ('bin_equal_width_10', 'education'), ('winsorize', 'education'), ('zscore_clip_3', 'education'), ('norm_log', 'married'))
--------------------------------------------------
(('bin_equal_width_10', 'x1'), ('norm_log', 'x1'), ('bin_equal_frequency_2', 'x4'), ('bin_equal_width_2', 'x5'))


In [9]:
print(ate_bins_data_lalonde.iloc[0]['best_sequence'])


(('zscore_clip_3', 'age'), ('bin_equal_frequency_10', 'age'), ('zscore_filter_3', 'education'), ('bin_equal_frequency_2', 'education'), ('isolationForest', 'age'))


In [16]:
sequence = (('zscore_clip_3', 'age'), ('bin_equal_frequency_10', 'age'), ('zscore_filter_3', 'education'), ('bin_equal_frequency_2', 'education'), ('isolationForest', 'age'))
transformed_df = apply_data_preparations_seq(df_lalonde.copy(), sequence, largest_data_transformations)

print(transformed_df[['black', 'hispanic', 'married']].describe())
print(transformed_df.shape)
transformed_df.columns[transformed_df.nunique() == 1].tolist()

            black    hispanic     married
count  438.000000  438.000000  438.000000
mean     0.833333    0.086758    0.168950
std      0.373104    0.281802    0.375136
min      0.000000    0.000000    0.000000
25%      1.000000    0.000000    0.000000
50%      1.000000    0.000000    0.000000
75%      1.000000    0.000000    0.000000
max      1.000000    1.000000    1.000000
(438, 8)


In [38]:
for s in ate_bins_data_lalonde['best_sequence'].head(5):
    print(s)
    print("~"*100)

(('zscore_clip_3', 'age'), ('bin_equal_frequency_10', 'age'), ('zscore_filter_3', 'education'), ('bin_equal_frequency_2', 'education'), ('isolationForest', 'age'))
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
(('norm_log', 'age'), ('zscore_clip_3', 'age'), ('bin_equal_frequency_10', 'age'), ('bin_equal_width_2', 'education'), ('isolationForest', 'age'))
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
(('zscore_clip_3', 'age'), ('bin_equal_frequency_10', 'age'), ('IQR', 'education'), ('bin_equal_frequency_2', 'education'), ('isolationForest', 'age'))
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
(('bin_equal_frequency_10', 'age'), ('zscore_filter_3', 'education'), ('bin_equal_frequency_2', 'education'), ('isolationForest', 'age'))
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [36]:
ate_bins_data_lalonde.head(5)

,bucket_range,min_ate_in_bin,max_ate_in_bin,min_length,best_sequence,count
0,"(-29.78, -25.079]",-29.780311,-29.780311,5,"(('zscore_clip_3', 'age'), ('bin_equal_frequen...",1
1,"(346.29, 350.991]",350.882028,350.882028,5,"(('norm_log', 'age'), ('zscore_clip_3', 'age')...",1
2,"(383.897, 388.598]",388.334061,388.334061,5,"(('zscore_clip_3', 'age'), ('bin_equal_frequen...",1
3,"(388.598, 393.299]",390.078195,390.078195,4,"(('bin_equal_frequency_10', 'age'), ('zscore_f...",8
4,"(393.299, 398.0]",393.525670,393.525670,5,"(('bin_equal_frequency_10', 'age'), ('zscore_f...",1


ASSES EXPERIMENTS

In [8]:



# def iter_out_files(path):
#     """
#     Yield (filename, content) from either:
#       - a .tar.gz/.tgz archive
#       - a directory containing extracted files
#     """
#     path = Path(path)
#
#     if path.is_dir():
#         # Walk recursively through extracted directory
#         for out_file in path.rglob("*.out"):
#             yield out_file.name, out_file.read_text(errors="ignore")
#
#     elif tarfile.is_tarfile(path):
#         with tarfile.open(path, "r:*") as tar:   # supports gz, bz2, xz, etc.
#             for member in tar.getmembers():
#                 if not member.isfile() or not member.name.endswith(".out"):
#                     continue
#
#                 f = tar.extractfile(member)
#                 if f is not None:
#                     yield member.name, f.read().decode("utf-8", errors="ignore")
#
#     else:
#         raise ValueError(f"'{path}' is neither a directory nor a tar archive.")

def iter_out_files(path):
    """
    Yield (filename, content) from either:
      - a .tar.gz/.tgz/.tar archive
      - a directory containing extracted files
    Accepts both .out and .txt files.
    """
    path = Path(path)
    # Define acceptable extensions as a tuple
    valid_extensions = (".out", ".txt")

    if path.is_dir():
        # Walk recursively through extracted directory
        for file in path.rglob("*"):
            if file.is_file() and file.suffix.lower() in valid_extensions:
                yield file.name, file.read_text(errors="ignore")

    elif tarfile.is_tarfile(path):
        with tarfile.open(path, "r:*") as tar:   # supports tar, gz, bz2, xz, etc.
            for member in tar.getmembers():
                # Check if it is a file and ends with .out or .txt
                if not member.isfile() or not member.name.lower().endswith(valid_extensions):
                    continue

                f = tar.extractfile(member)
                if f is not None:
                    yield member.name, f.read().decode("utf-8", errors="ignore")

    else:
        raise ValueError(f"'{path}' is neither a directory nor a tar archive.")


def print_full_raw_logs(source):
    # Grouping: exp_name -> exp_type -> run_number -> full_raw_text
    grouped_data = defaultdict(lambda: defaultdict(dict))

    # Regex patterns for grouping and metric extraction
    start_pattern = re.compile(r"Starting\s+(?P<exp>\w+),\s+type=(?P<type>\w+),\s+run\s+(?P<run>\d+)/3")

    time_pat = re.compile(r"Execution time:\s*([\d\.]+)\s*(?:seconds|sec)?", re.IGNORECASE)
    popped_pat = re.compile(r"popped\s+(\d+)\s+from Q", re.IGNORECASE)
    checked_pat1 = re.compile(r"checked\s+(\d+)\s+combinations", re.IGNORECASE)
    checked_pat2 = re.compile(r"checked:\n(\d+)", re.IGNORECASE)
    seq_pat = re.compile(r"(?:sequence is:|Most probable sequence:)\s*(.*)", re.IGNORECASE)
    ate_now_pat = re.compile(r"ATE now is:\s*([\d\.]+)", re.IGNORECASE)

    prob_pat = re.compile(r"((?:\([^)]+\))?\s*probability of this sequence is:\s*[\d\.e\-]+)", re.IGNORECASE)
    uncertainty_pat = re.compile(r"uncertainty:\s*\n\s*(\{.*?\})", re.IGNORECASE)

    # Works for both archives and folders
    for filename, content in iter_out_files(source):
        match = start_pattern.search(content)
        if match:
            exp = match.group("exp")
            exp_type = match.group("type")
            run = match.group("run")
            grouped_data[exp][exp_type][run] = content

    if not grouped_data:
        print("No matching .out files found.")
        return

    metrics_order = ['Status', 'Sequence', 'ATE Now', 'Probability',
                     'Uncertainty', 'Time', 'Popped', 'Checked']

    for exp, types in grouped_data.items():
        for exp_type, runs in types.items():
            print("~" * 90)
            print(f"{exp} | {exp_type}")
            print("-" * 90)

            run_metrics = {}
            sorted_run_nums = sorted(runs.keys())

            for run_num in sorted_run_nums:
                raw_text = runs[run_num]
                m = {}

                m['Status'] = "TIMEOUT" if "*** TIMED OUT!! ***" in raw_text else "FINISHED"

                t_match = time_pat.search(raw_text)
                if t_match:
                    time_val = float(t_match.group(1))
                    m['Time'] = f"{time_val:.3f}s"
                    m['Raw_Time'] = time_val
                else:
                    m['Time'] = "N/A"
                    m['Raw_Time'] = None

                pop_match = popped_pat.search(raw_text)
                m['Popped'] = pop_match.group(1) if pop_match else "N/A"

                check_match = checked_pat1.search(raw_text) or checked_pat2.search(raw_text)
                m['Checked'] = check_match.group(1) if check_match else "N/A"

                seq_match = seq_pat.search(raw_text)
                m['Sequence'] = seq_match.group(1).strip() if seq_match else "N/A"

                ate_n_match = ate_now_pat.search(raw_text)
                m['ATE Now'] = f"{float(ate_n_match.group(1)):.6f}" if ate_n_match else "N/A"

                prob_matches = prob_pat.findall(raw_text)
                m['Probability'] = " / ".join(p.strip() for p in prob_matches) if prob_matches else "N/A"

                unc_match = uncertainty_pat.search(raw_text)
                if unc_match:
                    clean_unc = (
                        unc_match.group(1)
                        .replace("np.float64(", "")
                        .replace("np.True_", "True")
                        .replace("np.False_", "False")
                        .replace(")", "")
                    )
                    m['Uncertainty'] = clean_unc
                else:
                    m['Uncertainty'] = "N/A"

                run_metrics[run_num] = m

            for key in metrics_order:
                values = [run_metrics[r][key] for r in sorted_run_nums]

                if all(v == "N/A" for v in values):
                    continue

                unique_vals = set(values)

                if key == 'Time':
                    valid_times = [
                        run_metrics[r]['Raw_Time']
                        for r in sorted_run_nums
                        if run_metrics[r]['Raw_Time'] is not None
                    ]
                    mean_suffix = (
                        f" (Mean: {sum(valid_times)/len(valid_times):.3f}s)"
                        if valid_times else ""
                    )

                    if len(unique_vals) == 1:
                        print(f"  • {key:<12}: {values[0]}{mean_suffix}")
                    else:
                        run_strings = [
                            f"Run {r}: {run_metrics[r][key]}"
                            for r in sorted_run_nums
                        ]
                        print(f"  • {key:<12}: {' | '.join(run_strings)}{mean_suffix}")
                else:
                    if len(unique_vals) == 1:
                        print(f"  • {key:<12}: {values[0]}")
                    else:
                        run_strings = [
                            f"Run {r}: {run_metrics[r][key]}"
                            for r in sorted_run_nums
                        ]
                        print(f"  • {key:<12}: {' | '.join(run_strings)}")

            print()
# print_full_raw_logs("acs_twins_run_28_05.tar.gz")
# print_full_raw_logs("exp_walmart.tar.gz")
# print_full_raw_logs("scale_exp")
# print_full_raw_logs("exp_27_28_long_TO.tar.gz")
print_full_raw_logs("exp_basic_larger.tar.gz")

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
EXP27 | brute
------------------------------------------------------------------------------------------
  • Status      : FINISHED
  • Sequence    : (('bin_equal_frequency_2', 'wt'), ('norm_log', 'gestat10'))
  • Time        : Run 1: 703.690s | Run 2: 692.537s | Run 3: 711.577s (Mean: 702.601s)
  • Popped      : 31331
  • Checked     : 18901165

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
EXP27 | probe_probs_no_restart
------------------------------------------------------------------------------------------
  • Status      : TIMEOUT

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
EXP27 | probe_probs_with_restart
------------------------------------------------------------------------------------------
  • Status      : TIMEOUT

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [168]:
from experiments import prob_dict, largest_data_transformations


def get_prob(sequence, trans_dict, op_probs, common_causes):
    pm = ProbManager([func_name for func_name, func in trans_dict.items()], common_causes, op_probs)
    return pm.get_sequence_probability(sequence)


seq =(('bin_equal_width_5', 'sqft_basement'), ('bin_equal_frequency_2', 'grade'), ('bin_equal_frequency_2', 'bathrooms'), ('bin_equal_frequency_5', 'sqft_living'), ('bin_equal_frequency_2', 'sqft_lot15'), ('bin_equal_frequency_2', 'yr_built'), ('bin_equal_frequency_2', 'sqft_living15'))





prob = get_prob(seq, largest_data_transformations, prob_dict, common_causes_walmart)
print(prob)
#
# seqx = [(('zscore_clip_3', 'private health coverage'), ('bin_equal_width_2', 'Age'), ('bin_equal_frequency_10', 'gender')),
# (('bin_equal_frequency_5', 'insurance through employer'), ('bin_equal_width_2', 'Public health coverage'), ('bin_equal_frequency_5', 'education')),
# (('norm_min_max', 'medicare for people 65 and older'), ('zscore_filter_3', 'insurance through employer'), ('bin_equal_width_5', 'insurance through employer')),
# (('norm_log', 'medicare for people 65 and older'), ('norm_min_max', 'gender'), ('bin_equal_width_2', 'education')),
# (('bin_equal_width_2', 'Public health coverage'), ('bin_equal_width_5', 'insurance through employer'), ('IQR', 'Age')),
# (('bin_equal_frequency_5', 'medicare for people 65 and older'), ('zscore_clip_3', 'Public health coverage'), ('bin_equal_frequency_2', 'gender')),
# (('isolationForest', 'education'), ('IQR', 'gender'), ('zscore_clip_3', 'Age')),
# (('bin_equal_frequency_10', 'education'), ('IQR', 'private health coverage'), ('winsorize', 'Age')),
# (('IQR', 'education'), ('bin_equal_frequency_10', 'medicare for people 65 and older'), ('bin_equal_frequency_10', 'private health coverage')),
# (('IQR', 'gender'), ('bin_equal_frequency_10', 'insurance through employer'), ('norm_log', 'insurance through employer'))]
# probs = []
# for s in seqx:
#   probs.append(get_prob(s, largest_data_transformations, prob_dict, common_causes_acs))
#
# print(np.mean(np.array(probs)))
# print(probs)


6.169639005938511e-63


In [45]:
def get_ci(orig_df, seq, transformations_dict, common_causes):
    curr_df = apply_data_preparations_seq(orig_df.copy(), seq, transformations_dict)
    return calculate_ate_with_uncertainty(curr_df.copy(), 'treatment', 'outcome', common_causes)['ci']

seq =(('winsorize', 'bathrooms'), ('winsorize', 'bedrooms'), ('winsorize', 'condition'), ('winsorize', 'floors'), ('winsorize', 'grade'), ('winsorize', 'sqft_above'), ('winsorize', 'sqft_basement'), ('winsorize', 'sqft_living'), ('winsorize', 'sqft_living15'), ('winsorize', 'sqft_lot'), ('winsorize', 'sqft_lot15'), ('winsorize', 'view'), ('winsorize', 'waterfront'), ('winsorize', 'yr_built'), ('winsorize', 'yr_renovated'))



print(get_ci(df_walmart, seq, largest_data_transformations, common_causes_walmart))

(np.float64(-78.25171096804843), np.float64(108947.61224775352))


In [35]:
import time
import re
import json
import subprocess
import numpy as np
from collections import Counter

from experiments import prob_dict
from search_methods.probe_ATE_search import ProbManager


N_RUNS = 10

experiments = [23, 25, 35]# experiments = [23, 25, 35]
functions = [
    "llm_zero_shot",
    "llm_few_shot",
    "llm_few_shot_cot"
]


ate_pattern = re.compile(r"ATE IS:\s*([-+]?\d*\.?\d+)")


def extract_sequence(output):

    start = output.find("[")

    if start == -1:
        return None

    depth = 0
    end = None

    for i in range(start, len(output)):

        if output[i] == "[":
            depth += 1

        elif output[i] == "]":
            depth -= 1

            if depth == 0:
                end = i + 1
                break

    if end is None:
        return None

    try:
        return json.loads(output[start:end])

    except Exception:
        return None



def get_prob(sequence, trans_dict, op_probs, common_causes):

    pm = ProbManager(
        [func_name for func_name, func in trans_dict.items()],
        common_causes,
        op_probs
    )

    return pm.get_sequence_probability(sequence)



for exp in experiments:

    print("\n" + "=" * 60)
    print(f" 🚀 STARTING EXPERIMENT: EXP{exp}")
    print("=" * 60)


    if exp == 23 or exp == 27:
        common_causes_exp = common_causes_twins
    if exp == 25 or exp == 28:
        common_causes_exp = common_causes_acs
    if exp == 35:
        common_causes_exp = common_causes_walmart

    for level in ["low"]:#, "high"]:

        print(f"\n 📊 Level: {level.upper()}")
        print("-" * 40)

        for func in functions:

            exp_arg = f"EXP{exp}_{level}" if (exp !=35 and exp !=27 and exp !=28) else f"EXP{exp}"
            print(
                f"\n⚙️ Running {func} ({N_RUNS} repetitions)"
            )


            run_data = []


            for run_idx in range(N_RUNS):

                start_time = time.time()


                result = subprocess.run(
                    [
                        "python",
                        "main.py",
                        exp_arg,
                        func
                    ],
                    capture_output=True,
                    text=True
                )


                output = result.stdout


                elapsed = time.time() - start_time


                ate_match = ate_pattern.search(output)


                if ate_match:

                    ate = float(ate_match.group(1))

                    # round once and use this everywhere
                    ate = round(ate, 5)


                    sequence = extract_sequence(output)


                    run_data.append(
                        {
                            "ate": ate,
                            "sequence": sequence,
                            "runtime": elapsed
                        }
                    )


                    print(
                        f"  Run {run_idx+1}/{N_RUNS}: "
                        f"ATE={ate}"
                    )

                else:

                    print(
                        f"  Run {run_idx+1}/{N_RUNS}: "
                        "ATE not found"
                    )


                print(
                    f"    Runtime: {elapsed:.2f}s"
                )


                time.sleep(7)



            # ===========================
            # Results
            # ===========================

            if run_data:


                ate_values = [
                    x["ate"]
                    for x in run_data
                ]


                runtimes = [
                    x["runtime"]
                    for x in run_data
                ]


                mean_ate = np.mean(ate_values)


                variance_ate = (
                    np.var(
                        ate_values,
                        ddof=1
                    )
                    if len(ate_values) > 1
                    else 0
                )


                std_ate = (
                    np.std(
                        ate_values,
                        ddof=1
                    )
                    if len(ate_values) > 1
                    else 0
                )


                avg_runtime = np.mean(runtimes)



                # ---------------------------
                # Most frequent ATE
                # ---------------------------

                ate_counter = Counter(
                    ate_values
                )


                most_common_ate, ate_freq = (
                    ate_counter.most_common(1)[0]
                )


                matching_runs = [
                    x
                    for x in run_data
                    if x["ate"] == most_common_ate
                ]


                chosen_sequence = (
                    matching_runs[0]["sequence"]
                    if matching_runs
                    else None
                )



                print("\nResults:")
                print(
                    f"  ATEs          : {ate_values}"
                )
                print(
                    f"  Mean ATE      : {mean_ate:.6f}"
                )
                print(
                    f"  Variance      : {variance_ate:.6f}"
                )
                print(
                    f"  Std Dev       : {std_ate:.6f}"
                )
                print(
                    f"  Avg Runtime   : {avg_runtime:.2f}s"
                )


                print("\nMost Frequent ATE:")
                print(
                    f"  ATE           : {most_common_ate:.5f}"
                )
                print(
                    f"  Frequency     : "
                    f"{ate_freq}/{N_RUNS}"
                )


                if chosen_sequence:


                    print(
                        "\nCorresponding Sequence:"
                    )

                    print(
                        json.dumps(
                            chosen_sequence,
                            indent=2
                        )
                    )



                    prob_sequence = tuple(
                        (
                            step["operation"],
                            step["column"]
                        )
                        for step in chosen_sequence
                    )


                    try:

                        sequence_probability = get_prob(
                            prob_sequence,
                            largest_data_transformations,
                            prob_dict,
                            common_causes_exp
                        )


                        print(
                            f"\nSequence Probability: "
                            f"{sequence_probability}"
                        )


                    except Exception as e:

                        print(
                            "\nCould not calculate "
                            "sequence probability:"
                        )

                        print(e)



                else:

                    print(
                        "\nNo valid sequence extracted"
                    )



    print("\n" + "=" * 60)
    print(
        f" ✅ FINISHED EXPERIMENT: EXP{exp}"
    )
    print("=" * 60)


 🚀 STARTING EXPERIMENT: EXP35

 📊 Level: LOW
----------------------------------------

⚙️ Running llm_zero_shot (10 repetitions)
  Run 1/10: ATE=54094.57986
    Runtime: 15.20s
  Run 2/10: ATE=54055.29509
    Runtime: 7.90s
  Run 3/10: ATE=54094.57986
    Runtime: 7.56s
  Run 4/10: ATE=54094.57986
    Runtime: 8.63s
  Run 5/10: ATE=44710.7626
    Runtime: 13.16s
  Run 6/10: ATE=54055.29509
    Runtime: 7.29s
  Run 7/10: ATE=49572.84707
    Runtime: 11.70s
  Run 8/10: ATE=45980.5313
    Runtime: 13.47s
  Run 9/10: ATE=54055.29509
    Runtime: 7.32s
  Run 10/10: ATE=46327.7627
    Runtime: 12.54s

Results:
  ATEs          : [54094.57986, 54055.29509, 54094.57986, 54094.57986, 44710.7626, 54055.29509, 49572.84707, 45980.5313, 54055.29509, 46327.7627]
  Mean ATE      : 51104.152852
  Variance      : 16137935.511316
  Std Dev       : 4017.204938
  Avg Runtime   : 10.48s

Most Frequent ATE:
  ATE           : 54094.57986
  Frequency     : 3/10

Corresponding Sequence:
[
  {
    "column": "sq

PROBABILTYS CORELATIONS:

In [68]:
my_probs = {'bin_equal_frequency_2': 1e-10, 'bin_equal_frequency_5': 0.016233766233766232,
             'bin_equal_frequency_10': 0.012987012987012988, 'bin_equal_width_2': 1e-10,
             'bin_equal_width_5': 0.006493506493506494, 'bin_equal_width_10': 1e-10, 'norm_min_max': 0.577922077922078,
             'norm_log': 0.22727272727272727, 'zscore_clip_3': 0.025974025974025976,
             'zscore_filter_3': 0.025974025974025976, 'winsorize': 1e-10, 'IQR': 0.09090909090909091,
             'isolationForest': 0.04220779220779221}

my_probs_adjusted = {
    "log_transform": 0.22727272727272727,
    "min_max_scaling": 0.577922077922078,
    "iqr_filter":  0.09090909090909091,
    "zscore_filter_clip": 0.05194805194805195,
    "equal_freq_binning": 0.02922077932077922,
    "winsorize": 1e-10,
    "isolation_forest": 0.04220779220779221,
    "equal_width_binning": 0.006493506693506493
}

data_prep_probabilities = { # GEMINI 3.1 pro
    "log_transform": 0.28,
    "min_max_scaling": 0.22,
    "iqr_filter": 0.18,
    "zscore_filter_clip": 0.14,
    "equal_freq_binning": 0.07,
    "winsorize": 0.06,
    "isolation_forest": 0.03,
    "equal_width_binning": 0.02
}

s_mine = pd.Series(my_probs_adjusted)
s_gemini = pd.Series(data_prep_probabilities)

print(f"Correlation is {s_mine.corr(s_gemini)}")

Correlation is 0.6867487300014808


Scale experiments

In [6]:

import json
import numpy as np

def parse_standalone_content(content):
    # Match: "Starting EXP27, type=probe, run 1/3"
    meta_match = re.search(r"Starting\s+EXP([\d.]+),\s*type=([\w.-]+).*?run\s+(\d+)", content, re.IGNORECASE)
    if not meta_match:
        return None, None, None, None

    exp_id = meta_match.group(1)
    exp_type = meta_match.group(2)
    run_num = int(meta_match.group(3))

    # We only want whole numbers in this script
    if "." in exp_id:
        return None, None, None, None

    status = "TIMEOUT" if "TIME OUT" in content else "FINISHED"

    time_match = re.search(r"Execution time:\s*([\d.]+)\s*sec", content)
    exec_time = float(time_match.group(1)) if time_match and status == "FINISHED" else None

    distances = []
    dist_match = re.search(r"distances from ATE \(with time\):\s*\n(\[.*?\])", content, re.DOTALL)
    if dist_match and status == "FINISHED":
        list_str = re.sub(r"np\.float64\((.*?)\)", r"\1", dist_match.group(1))
        try:
            distances = ast.literal_eval(list_str)
        except Exception:
            pass

    return exp_id, exp_type, run_num, {"status": status, "time": exec_time, "distances": distances}

def process_standalone(folder_path):
    # Hierarchy: { "27": { "probe": { 1: data, 2: data, 3: data } } }
    raw_results = defaultdict(lambda: defaultdict(dict))

    dir_path = Path(folder_path)
    print(f"Reading folder '{dir_path}' for Standalone Experiments...")

    # Look for files ending in either .out or .err inside the folder and subfolders
    for file_path in dir_path.rglob("*"):
        if file_path.is_file() and file_path.suffix in (".out", ".err"):
            # Read and decode the file contents directly from disk
            content = file_path.read_text(encoding="utf-8", errors="ignore")

            exp_id, exp_type, run_num, run_data = parse_standalone_content(
                content
            )

            if exp_id:
                raw_results[exp_id][exp_type][run_num] = run_data

    final_output = {}
    for exp_id, types_dict in raw_results.items():
        exp_key = f"EXP{exp_id}"
        final_output[exp_key] = {}

        for exp_type, runs_dict in types_dict.items():
            # Ensure order: run 1, run 2, run 3
            sorted_runs = [
                runs_dict.get(
                    i, {"status": "MISSING", "time": None, "distances": []}
                )
                for i in range(1, 4)
            ]

            exec_times = [r["time"] for r in sorted_runs if r["time"] is not None]
            mean_time = float(np.mean(exec_times)) if exec_times else None
            all_distances = [r["distances"] for r in sorted_runs]
            statuses = [r["status"] for r in sorted_runs]

            final_output[exp_key][exp_type] = {
                "run_statuses": statuses,
                "execution_times": exec_times,
                "mean_execution_time": mean_time,
                "distances_per_run": all_distances,
            }

    return final_output


if __name__ == "__main__":
    TAR_FILE_PATH = "scale_exp"
    results = process_standalone(TAR_FILE_PATH)

    with open("standalone_results.json", "w") as f:
        json.dump(results, f, indent=4)
    print("Saved standalone_results.json")

Reading folder 'scale_exp' for Standalone Experiments...
Saved standalone_results.json


In [20]:
results["EXP35"]

{'probe': {'run_statuses': ['FINISHED', 'FINISHED', 'FINISHED'],
  'execution_times': [299.925, 294.455, 298.752],
  'mean_execution_time': 297.71066666666667,
  'distances_per_run': [[(51434.51193967457, 0),
    (49623.77463243274, 0.009455442428588867),
    (51434.51193967457, 0.013483285903930664),
    (49623.77463243274, 0.0245058536529541),
    (43096.097209752945, 0.034895896911621094),
    (51434.51193967457, 0.03891253471374512),
    (49623.77463243274, 0.049636125564575195),
    (43096.097209752945, 0.0600886344909668),
    (41449.12177303797, 0.19752120971679688),
    (40312.08393973469, 0.29555654525756836),
    (35288.88453374006, 0.31777024269104004),
    (51434.51193967457, 0.32216548919677734),
    (49623.77463243274, 0.33495211601257324),
    (43096.097209752945, 0.3469047546386719),
    (35288.88453374006, 0.3571150302886963),
    (35187.66428235199, 0.3832695484161377),
    (33087.27912870823, 0.4266202449798584),
    (32086.269298315623, 0.7664306163787842),
    (317

In [7]:
results_ablation_twins = process_standalone('ablation_twins (1)')
with open("ablation_twins.json", "w") as f:
    json.dump(results_ablation_twins, f, indent=4)
print("Saved ablation twins.json")

results_ablation_acs = process_standalone('ablation_acs')
with open("ablation_acs.json", "w") as f:
    json.dump(results_ablation_acs, f, indent=4)
print("Saved ablation acs.json")

results_ablation_walmart = process_standalone('ablation_walmart')
with open("ablation_walmart.json", "w") as f:
    json.dump(results_ablation_walmart, f, indent=4)
print("Saved ablation walmart.json")

Reading folder 'ablation_twins (1)' for Standalone Experiments...
Saved ablation twins.json
Reading folder 'ablation_acs' for Standalone Experiments...
Saved ablation acs.json
Reading folder 'ablation_walmart' for Standalone Experiments...
Saved ablation walmart.json


In [17]:
results_ablation_walmart

{'EXP35': {'probe_no_hash': {'run_statuses': ['FINISHED',
    'FINISHED',
    'FINISHED'],
   'execution_times': [1818.295, 1823.79, 1837.629],
   'mean_execution_time': 1826.5713333333333,
   'distances_per_run': [[(51434.51195990761, 0),
     (49623.77463746477, 0.007904291152954102),
     (51434.51195990761, 0.008511543273925781),
     (49623.77463746477, 0.015547513961791992),
     (43096.09723924365, 0.0208282470703125),
     (51434.51195990761, 0.021426677703857422),
     (49623.77463746477, 0.026231765747070312),
     (43096.09723924365, 0.030800342559814453),
     (41449.12179651396, 0.09703731536865234),
     (40312.083954030764, 0.14127111434936523),
     (35288.88450313075, 0.153395414352417),
     (51434.51195990761, 0.15683341026306152),
     (49623.77463746477, 0.16193032264709473),
     (43096.09723924365, 0.16660308837890625),
     (35288.88450313075, 0.1716153621673584),
     (35187.66430007291, 0.18662595748901367),
     (33087.279112568365, 0.21247458457946777),
    

In [63]:
from pathlib import Path
import json
from collections import defaultdict

# Change this value to whatever placeholder you prefer (e.g., -1, 99999, or "TIMEOUT")
TIMEOUT_VAL = 14400


def parse_dotted_content(content):
    # Match: "Starting EXP30.1, type=probe, run 1/3"
    meta_match = re.search(
        r"Starting\s+EXP([\d.]+),\s*type=([\w.-]+).*?run\s+(\d+)",
        content,
        re.IGNORECASE,
    )
    if not meta_match:
        return None, None, None, None, None

    full_exp_id = meta_match.group(1)
    exp_type = meta_match.group(2)
    run_num = int(meta_match.group(3))

    # We ONLY want dotted numbers in this script
    if "." not in full_exp_id:
        return None, None, None, None, None

    parent_id = full_exp_id.split(".")[0]  # "30.1" -> "30"

    if "TIME OUT" in content:
        return parent_id, full_exp_id, exp_type, run_num, TIMEOUT_VAL

    time_match = re.search(r"Execution time:\s*([\d.]+)\s*sec", content)
    exec_time = float(time_match.group(1)) if time_match else TIMEOUT_VAL

    return parent_id, full_exp_id, exp_type, run_num, exec_time


def process_dotted(folder_path):
    # Hierarchy: { "30": { "probe": { "30.1": {1: time, 2: time, 3: time}, "30.2": {...} } } }
    raw_results = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

    dir_path = Path(folder_path)
    print(f"Reading folder '{dir_path}' for Dotted Sub-experiments...")

    # Look for files ending in either .out or .err inside the folder and subfolders
    for file_path in dir_path.rglob("*"):
        if file_path.is_file() and file_path.suffix in (".out", ".err"):
            # Read and decode the file contents directly from disk
            content = file_path.read_text(encoding="utf-8", errors="ignore")

            parent_id, full_id, exp_type, run_num, exec_time = (
                parse_dotted_content(content)
            )

            if parent_id:
                raw_results[parent_id][exp_type][full_id][run_num] = exec_time

    final_output = {}
    for parent_id, types_dict in raw_results.items():
        parent_key = f"EXP{parent_id}"
        final_output[parent_key] = {}

        for exp_type, sub_exps in types_dict.items():
            # Sort sub-experiments numerically so 30.1 comes before 30.2
            sorted_sub_exp_keys = sorted(sub_exps.keys(), key=lambda x: float(x))

            type_run_lists = []
            for sub_key in sorted_sub_exp_keys:
                runs_dict = sub_exps[sub_key]
                # Keep exactly 3 entries. If a run doesn't exist or timed out, use TIMEOUT_VAL
                times = [runs_dict.get(i, TIMEOUT_VAL) for i in range(1, 4)]
                type_run_lists.append(times)

            final_output[parent_key][exp_type] = type_run_lists

    return final_output


if __name__ == "__main__":
    TAR_FILE_PATH = "scale_exp"
    results = process_dotted(TAR_FILE_PATH)

    with open("dotted_results.json", "w") as f:
        json.dump(results, f, indent=4)
    print("Saved dotted_results.json")

Reading folder 'exp_F' for Dotted Sub-experiments...


In [64]:
results

{'EXP41': {'fprobe': [[119.875, 119.84, 120.706],
   [229.269, 229.083, 231.449],
   [222.026, 222.568, 222.226],
   [221.948, 221.486, 191.22],
   [195.307, 196.338, 194.867],
   [248.431, 248.837, 249.091],
   [250.74, 253.038, 252.201],
   [171.276, 222.096, 221.656],
   [187.12, 185.898, 187.025],
   [195.148, 193.259, 192.829],
   [191.217, 188.314, 189.374],
   [224.537, 224.764, 215.744],
   [219.945, 219.28, 219.056],
   [224.164, 224.371, 222.856],
   [227.929, 226.141, 224.535]]},
 'EXP43': {'fprobe_probs_no_restart': [[2171.771, 2171.548, 2145.066],
   [501.967, 500.661, 499.631],
   [14400, 14400, 14400],
   [14400, 14400, 14400],
   [3088.622, 3028.213, 3035.802],
   [3284.503, 3281.172, 3294.014],
   [14400, 14400, 14400],
   [766.887, 771.196, 770.56],
   [1167.119, 1173.885, 1167.99],
   [14400, 14400, 14400],
   [14400, 14400, 14400],
   [14400, 14400, 14400],
   [2199.849, 2213.679, 2227.694],
   [2227.081, 2245.978, 2229.971],
   [2499.73, 2491.91, 2680.47]]},
 'EXP4

In [22]:
results.keys()

dict_keys(['EXP30', 'EXP29', 'EXP31', 'EXP32', 'EXP33', 'EXP34', 'EXP36', 'EXP37', 'EXP38'])

DR & DML

In [1]:
# from utils import  calculate_ate_dml, calculate_ate_dr  # manual_dr_ate, manual_dml_ate
#
#
# # =========================
# # Prepare datasets
# # =========================
#
# df_acs_copy = df_acs.copy()
# df_acs_copy['treatment'] = df_acs_copy['treatment'] - 1
#
#
# # =========================
# # Data preparation sequences
# # =========================
#
# sequence_twins = (
#     ('bin_equal_frequency_2', 'wt'),
#     ('norm_log', 'gestat10')
# )
#
# sequence_acs = (
#     ('bin_equal_width_2', 'education'),
#     ('isolationForest', 'Age'),
#     ('bin_equal_frequency_2', 'Age')
# )
#
# sequence_walmart = (
#     (('bin_equal_width_2', 'sqft_basement'), ('bin_equal_frequency_10', 'sqft_living'), ('bin_equal_frequency_2', 'yr_built'), ('bin_equal_frequency_2', 'sqft_living15'), ('bin_equal_frequency_2', 'sqft_lot15'))
# )
#
#
# # =========================
# # Apply transformations
# # =========================
#
# # transformed_df_twins = apply_data_preparations_seq(
# #     df_twins,
# #     sequence_twins,
# #     largest_data_transformations
# # )
#
# transformed_df_acs = apply_data_preparations_seq(
#     df_acs_copy,
#     sequence_acs,
#     largest_data_transformations
# )
#
# transformed_df_walmart = apply_data_preparations_seq(
#     df_walmart,
#     sequence_walmart,
#     largest_data_transformations
# )
#
#
# # =========================
# # Print results
# # =========================
#
# # print(
# #     f"twins\n"
# #     f"start ATE:\n"
# #     f"DR:\n"
# #     f"before: {ate_dr(df_twins)} | after: {ate_dr(transformed_df_twins)}\n"
# #     f"DML:\n"
# #     f"before: {ate_dml(df_twins)} | after: {ate_dml(transformed_df_twins)}"
# # )
#
# print("~" * 120)
#
# print(
#     f"acs\n"
#     f"start ATE:\n"
#     f"DR:\n"
#     f"before: {calculate_ate_dr(df_acs_copy)} | after: {calculate_ate_dr(transformed_df_acs)}\n"
#     f"DML:\n"
#     f"before: {calculate_ate_dml(df_acs_copy)} | after: {calculate_ate_dml(transformed_df_acs)}"
# )
#
# print("~" * 120)
#
# print(
#     f"walmart\n"
#     f"start ATE:\n"
#     f"DR:\n"
#     f"before: {calculate_ate_dr(df_walmart)} | after: {calculate_ate_dr(transformed_df_walmart)}\n"
#     f"DML:\n"
#     f"before: {calculate_ate_dml(df_walmart)} | after: {calculate_ate_dml(transformed_df_walmart)}"
# )

In [99]:
# from econml.dml import LinearDML
# from sklearn.linear_model import LassoCV
#
# # 1. Define the DML estimator
# est = LinearDML(
#     model_y=LassoCV(cv=3),  # Model for outcome
#     model_t=LassoCV(cv=3),  # Model for treatment
#     cv=2,                   # 2-fold cross-fitting
#     discrete_treatment=False
# )
#
# # 2. Fit to data (X=covariates, T=treatment, Y=outcome)
# est.fit(Y=df_acs_copy['outcome'], T=df_acs_copy['treatment'], X=df_acs_copy.drop(columns=['outcome', 'treatment']))
#
# # 3. Get ATE and statistical inference
# ate = est.ate(X=df_acs_copy.drop(columns=['outcome', 'treatment']))
# # stderr = est.ate_stderr(X=df_acs_copy.drop(columns=['outcome', 'treatment']))
# ci = est.ate_interval(X=df_acs_copy.drop(columns=['outcome', 'treatment']))
#
# print(f"ATE: {ate:.4f}, 95% CI: {ci})")


In [95]:
# # print(df_acs_copy['treatment'].value_counts(dropna=False))
# df_acs_copy.isna().sum().sum()
# df_acs_copy.drop(columns=['outcome', 'treatment']).isna().sum().sum()

np.int64(0)

In [20]:
acs_df_1 = df_acs.copy()
acs_df_2 =pd.concat([ df_acs.copy()] * 2, ignore_index=True)

print(f"ATE acs*1: {calculate_ate_with_uncertainty(acs_df_1, 'treatment', 'outcome', common_causes_acs)['ate']}")
print(f"ATE acs*2: {calculate_ate_with_uncertainty(acs_df_2, 'treatment', 'outcome', common_causes_acs)['ate']}")

print("lets do ISOLATION FOREST")

print(f"ATE acs*1: {calculate_ate_with_uncertainty(isolationForest(acs_df_1,'z'), 'treatment', 'outcome', common_causes_acs)['ate']}")
print(f"ATE acs*2: {calculate_ate_with_uncertainty(isolationForest(acs_df_2,'z'), 'treatment', 'outcome', common_causes_acs)['ate']}")



ATE acs*1: 8774.433204698473
ATE acs*2: 8774.43320467728
lets do ISOLATION FOREST
ATE acs*1: 14268.239362588889
ATE acs*2: 15183.847057103885


F experiment

In [143]:

import json


def parse_and_sort_experiment_archive(tar_path):
    # Example:
    # Starting EXP41.1, type=fprobe, run 1/3
    header_regex = re.compile(
        r"Starting EXP(\d+)\.(\d+),\s*type=([^,]+),\s*run (\d+)/(\d+)"
    )

    time_regex = re.compile(r"Execution time:\s*([\d\.]+)\s*sec")

    raw_results = {}

    # Read files directly from tar.gz
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():

            # Only process .out files
            if not member.isfile() or not member.name.endswith(".out"):
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            lines = f.read().decode("utf-8", errors="replace").splitlines(True)

            if not lines:
                continue

            first_line = lines[0].strip()

            match = header_regex.search(first_line)
            if not match:
                continue

            major_exp = match.group(1)
            sub_exp = match.group(2)
            run_type = match.group(3)
            run_idx = int(match.group(4)) - 1

            sub_exp_key = f"{major_exp}.{sub_exp}"

            # Create structure:
            # EXP -> sub EXP -> type -> [run1, run2, run3]
            raw_results.setdefault(major_exp, {})
            raw_results[major_exp].setdefault(sub_exp_key, {})
            raw_results[major_exp][sub_exp_key].setdefault(
                run_type,
                [None, None, None]
            )

            file_content = "".join(lines)

            if "TIMED OUT!" in file_content:
                run_result = "TIMEOUT"

            elif "FINISHED" in file_content:
                time_match = time_regex.search(file_content)
                run_result = (
                    float(time_match.group(1))
                    if time_match
                    else "FINISHED"
                )

            else:
                run_result = "UNKNOWN"

            if 0 <= run_idx < 3:
                raw_results[major_exp][sub_exp_key][run_type][run_idx] = run_result


    # Sort output:
    # EXP number -> sub experiment -> type alphabetically
    perfectly_sorted = {}

    for major in sorted(raw_results.keys(), key=int):
        perfectly_sorted[major] = {}

        sorted_subs = sorted(
            raw_results[major].keys(),
            key=lambda x: [int(num) for num in x.split(".")]
        )

        for sub in sorted_subs:
            perfectly_sorted[major][sub] = {}

            for run_type in sorted(raw_results[major][sub].keys()):
                perfectly_sorted[major][sub][run_type] = (
                    raw_results[major][sub][run_type]
                )

    return perfectly_sorted



if __name__ == "__main__":
    archive = "exp_F_2.tar.gz"#"./exp_F (1).tar.gz"#./exp_F.tar.gz"

    exp_data = parse_and_sort_experiment_archive(archive)

    print(json.dumps(exp_data, indent=4))

    with open("F_exp.json", "w") as f:
        json.dump(exp_data, f, indent=4)

    print("Saved F_exp.json")

    archive = "exp_f_walmart_no_rest.tar.gz"#"exp_F_2.tar.gz"#"./exp_F (1).tar.gz"#./exp_F.tar.gz"

    exp_data = parse_and_sort_experiment_archive(archive)

    print(json.dumps(exp_data, indent=4))

    with open("F_exp_walmart_no_restart.json", "w") as f:#with open("F_exp.json", "w") as f:
        json.dump(exp_data, f, indent=4)

    print("Saved exp_f_walmart_no_rest.json")

{
    "41": {
        "41.1": {
            "fprobe": [
                275.778,
                158.131,
                159.85
            ]
        },
        "41.2": {
            "fprobe": [
                232.665,
                233.496,
                231.8
            ]
        },
        "41.3": {
            "fprobe": [
                230.397,
                246.013,
                263.347
            ]
        },
        "41.4": {
            "fprobe": [
                250.129,
                249.912,
                249.511
            ]
        },
        "41.5": {
            "fprobe": [
                234.066,
                255.524,
                231.81
            ]
        },
        "41.6": {
            "fprobe": [
                250.86,
                234.374,
                247.669
            ]
        },
        "41.7": {
            "fprobe": [
                263.62,
                264.0,
                284.339
            ]
        },
       

In [148]:


def print_raw_out_files(run_id, tar_path="./exp_F.tar.gz"):
    """
    Scans './exp_F.tar.gz', parses the headers of the .out files,
    and prints the raw content of any file matching the requested run_id (e.g., "44.1").
    """
    # 1. Parse target major and minor from input "44.1" -> ("44", "1")
    try:
        target_major, target_minor = run_id.strip().split('.')
    except ValueError:
        print("❌ Error: Please provide the run ID in 'major.minor' format (e.g., '44.1')")
        return

    # Use the exact same header regex as our parser script
    header_regex = re.compile(
        r"Starting EXP(\d+)\.(\d+),\s*type=([^,]+),\s*run (\d+)/(\d+)"
    )

    # Fallback to locate the tar file if the path differs
    if not os.path.exists(tar_path):
        tar_files = [f for f in os.listdir('.') if f.endswith('.tar.gz')]
        if tar_files:
            tar_path = tar_files[0]
            print(f"⚠️ Could not find './exp_F.tar.gz'. Using detected archive: {tar_path}")
        else:
            print("❌ Error: No .tar.gz archive found in the current directory.")
            return

    print(f"📦 Opening archive: {tar_path}")
    print(f"🔍 Searching for files belonging to EXP {target_major}.{target_minor}...\n")

    match_count = 0

    # 2. Iterate through files in the tarball
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():
            # Only look at files ending with .out
            if not member.isfile() or not member.name.endswith(".out"):
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            # Read and decode contents safely
            content_bytes = f.read()
            content_str = content_bytes.decode("utf-8", errors="replace")
            lines = content_str.splitlines()

            if not lines:
                continue

            # Read the very first line to check the header match
            first_line = lines[0].strip()
            match = header_regex.search(first_line)
            if not match:
                continue

            major_exp = match.group(1)
            sub_exp = match.group(2)
            run_type = match.group(3)
            run_idx = match.group(4)
            total_runs = match.group(5)

            # 3. Print the match if it matches our targets
            if major_exp == target_major and sub_exp == target_minor:
                match_count += 1
                print("=" * 80)
                print(f"📂 File Path: {member.name}")
                print(f"🎯 Header   : EXP {major_exp}.{sub_exp} | Type: {run_type} | Run: {run_idx}/{total_runs}")
                print("=" * 80)
                print(content_str.strip())
                print("=" * 80)
                print("\n")

    if match_count == 0:
        print(f"❌ Finished scanning. Found 0 matching logs for '{run_id}'.")
    else:
        print(f"✅ Displayed {match_count} raw log file(s) for EXP {run_id}.")

if __name__ == "__main__":
    # Example usage:
    print_raw_out_files("44.6")#,"exp_F_walmart_no_rest.tar.gz")

📦 Opening archive: ./exp_F.tar.gz
🔍 Searching for files belonging to EXP 44.6...

📂 File Path: exp41_45_68288676_150.out
🎯 Header   : EXP 44.6 | Type: fprobe | Run: 1/3
Starting EXP44.6, type=fprobe, run 1/3
SLURM job id: 68291803, array task id: 150
loaded cached data
loaded cached data
loaded cached data
loaded cached data
loaded cached data
loaded cached data
RUNNING EXPERIMENT WITH i=6. G size is 72 + 5
START ATE IS: 61567.591959907615
PROBE TRIGGERED! Error reduced from inf to 47312.931 (ATE went to 57446.01118477756).
PROBE TRIGGERED! Error reduced from 47312.931 to 41449.122 (ATE went to 51582.20179651396).
PROBE TRIGGERED! Error reduced from 41449.122 to 36910.133 (ATE went to 47043.21272772279).
PROBE TRIGGERED! Error reduced from 36910.133 to 32086.269 (ATE went to 42219.349265116645).
PROBE TRIGGERED! Error reduced from 32086.269 to 20836.602 (ATE went to 30969.682450802196).
PROBE TRIGGERED! Error reduced from 20836.602 to 16034.892 (ATE went to 26167.972409159524).
PROBE T

In [155]:
print_raw_out_files("45.3")

📦 Opening archive: ./exp_F.tar.gz
🔍 Searching for files belonging to EXP 45.3...

📂 File Path: exp41_45_68288676_186.out
🎯 Header   : EXP 45.3 | Type: fprobe_probs_with_restart | Run: 1/3
Starting EXP45.3, type=fprobe_probs_with_restart, run 1/3
SLURM job id: 68292347, array task id: 186
loaded cached data
loaded cached data
loaded cached data
loaded cached data
loaded cached data
loaded cached data
RUNNING EXPERIMENT WITH i=3. G size is 36 + 7
START ATE IS: 61567.591959907615
PROBE TRIGGERED! Error reduced from inf to 51434.512 (ATE went to 61567.59198608658).
PROBE TRIGGERED! Error reduced from 51434.512 to 45752.787 (ATE went to 55885.86667331493).
PROBE TRIGGERED! Error reduced from 45752.787 to 40630.659 (ATE went to 50763.73880947745).
PROBE TRIGGERED! Error reduced from 40630.659 to 34486.573 (ATE went to 44619.653268265436).
PROBE TRIGGERED! Error reduced from 34486.573 to 31028.036 (ATE went to 41161.11574781825).
PROBE TRIGGERED! Error reduced from 31028.036 to 27514.426 (ATE

In [153]:
import os
import re
import ast
import tarfile
import pandas as pd

# 1. SET YOUR TAR.GZ FILE PATH HERE
TAR_PATH = "exp_F.tar.gz" #"exp_F_2.tar.gz"#'exp_F_walmart_no_rest.tar.gz'

runs = []

# 2. Open and read directly from the tar.gz archive
with tarfile.open(TAR_PATH, 'r:gz') as tar:
    for member in tar.getmembers():
        if not member.isfile():
            continue  # Skip directories

        # Read the file content in memory
        f = tar.extractfile(member)
        if f is None:
            continue

        content = f.read().decode('utf-8', errors='ignore')
        f.close()

        # Split if files are concatenated, otherwise process as a single run block
        chunks = [c for c in re.split(r'={10,}', content) if 'EXP' in c] or [content]

        for chunk in chunks:
            # Experiment ID & Run Number
            exp_match = re.search(r'EXP\s*([\d.]+)', chunk, re.IGNORECASE)
            if not exp_match:
                continue
            exp_id = f'EXP_{exp_match.group(1)}'

            run_match = re.search(r'Run:?\s*(\d+)/(\d+)', chunk, re.IGNORECASE)
            run_num = run_match.group(1) if run_match else 1

            # ATEs (with salvage fallback for truncated logs)
            start_ate_match = re.search(r'START ATE IS:\s*([\d.-]+)', chunk)
            start_ate = float(start_ate_match.group(1)) if start_ate_match else None

            final_ate_match = re.search(r'ATE now is:\s*([\d.-]+)', chunk)
            finished = False
            if final_ate_match:
                ate = float(final_ate_match.group(1))
                finished = True
            else:
                # Salvage the last successful probe step's ATE before the crash/cutoff
                probe_ates = re.findall(r'ATE went to\s*([\d.inf.-]+)', chunk)
                valid_probes = [float(x) for x in probe_ates if x != 'inf']
                ate = valid_probes[-1] if valid_probes else start_ate

            # Pipeline Sequence (e.g., binning, scaling steps)
            seq_match = re.search(r'sequence is:\s*(.*)', chunk)
            sequence = None
            if seq_match:
                seq_str = seq_match.group(1).strip()
                try:
                    sequence = str(ast.literal_eval(seq_str))
                except Exception:
                    sequence = seq_str # Fallback to raw text if string is truncated

            # Sequence Probabilities (Extract both standard & real without collisions)
            prob = None
            real_prob = None
            for line in chunk.splitlines():
                if 'probability of this sequence is:' in line:
                    val_match = re.search(r'is:\s*([\d.e+-]+)', line, re.IGNORECASE)
                    if val_match:
                        val = float(val_match.group(1))
                        if 'REAL' in line:
                            real_prob = val
                        else:
                            prob = val

            # Execution Time
            exec_match = re.search(r'Execution time:\s*([\d.]+)\s*sec', chunk)
            exec_time = float(exec_match.group(1)) if exec_match else None

            # Uncertainty (Standard Error & Significance)
            se = None
            significant = None

            # Clean up numpy wrapper strings (e.g., np.float64, np.False_) for easy regexing
            cleaned_chunk = re.sub(r'np\.\w+\(([^)]+)\)', r'\1', chunk)
            cleaned_chunk = cleaned_chunk.replace('np.False_', 'False').replace('np.True_', 'True')

            se_match = re.search(r'se:\s*([\d.e+-]+)', cleaned_chunk)
            if se_match:
                try:
                    se = float(se_match.group(1))
                except ValueError:
                    pass
            sig_match = re.search(r'significant:\s*(\w+)', cleaned_chunk)
            if sig_match:
                significant = sig_match.group(1) == 'True'

            runs.append({
                'exp_id': exp_id,
                'run': run_num,
                'start_ate': start_ate,
                'final_ate': ate,
                'finished': finished,
                'sequence': sequence,
                'probability': prob,
                'real_probability': real_prob,
                'exec_time': exec_time,
                'se': se,
                'significant': significant
            })

# 3. Aggregate everything into a clean summary
if not runs:
    print(f"No experiment logs found in {TAR_PATH}")
else:
    df = pd.DataFrame(runs)
    summary = df.groupby('exp_id').agg(
        total_runs=('run', 'count'),
        completed_runs=('finished', 'sum'),
        start_ate=('start_ate', 'mean'),
        final_ate=('final_ate', 'mean'),
        avg_probability=('probability', 'mean'),
        avg_real_probability=('real_probability', 'mean'),
        avg_exec_time_sec=('exec_time', 'mean'),
        avg_se=('se', 'mean'),
        # Grab the pipeline sequence from whichever run completed/printed it first
        pipeline_sequence=('sequence', lambda x: x.dropna().iloc[0] if not x.dropna().empty else None),
        # Calculate the % of runs that returned a statistically significant result
        significance_rate=('significant', lambda x: x.dropna().mean() if not x.dropna().empty else None)
    ).reset_index()

    # Format significance rate as a clean percentage
    summary['significance_rate'] = summary['significance_rate'].apply(
        lambda x: f"{x * 100:.0f}%" if pd.notna(x) else "N/A"
    )

    summary.to_csv('summary.csv', index=False)

    print('\n' + '=' * 140)
    print(summary)
    print('=' * 140)
    print('\nSaved full consolidated results to summary.csv')



       exp_id  total_runs  completed_runs     start_ate     final_ate  \
0    EXP_41.1           3               3   8774.433205  19619.032573   
1   EXP_41.10           3               3   8774.433205  19619.032573   
2   EXP_41.11           3               3   8774.433205  19619.032573   
3   EXP_41.12           3               3   8774.433205  19619.032573   
4   EXP_41.13           3               3   8774.433205  19619.032573   
..        ...         ...             ...           ...           ...   
70   EXP_45.5           6               3  61567.591960  12194.232092   
71   EXP_45.6           6               3  61567.591960  12194.232092   
72   EXP_45.7           6               3  61567.591960  12194.232092   
73   EXP_45.8           6               3  61567.591960  12366.127914   
74   EXP_45.9           6               3  61567.591960  12614.167407   

    avg_probability  avg_real_probability  avg_exec_time_sec avg_se  \
0               NaN                   NaN         1

In [154]:
summary[summary['avg_real_probability'] > 3.4e-18].sort_values(by='avg_real_probability')

,exp_id,total_runs,completed_runs,start_ate,final_ate,avg_probability,avg_real_probability,avg_exec_time_sec,avg_se,pipeline_sequence,significance_rate
64,EXP_45.13,3,3,61567.591960,11815.474528,0.039457,3.420411e-18,241.384000,NaN,"(('bin_equal_frequency_10', 'sqft_living'), ('...",N/A
65,EXP_45.14,3,3,61567.591960,12458.727870,0.000011,3.420411e-18,644.971000,NaN,"(('norm_log', 'grade'), ('IQR', 'sqft_basement...",N/A
66,EXP_45.15,3,1,61567.591960,12458.727870,0.000011,3.420411e-18,1072.008000,NaN,"(('norm_log', 'grade'), ('IQR', 'sqft_basement...",N/A
73,EXP_45.8,6,3,61567.591960,12366.127914,0.000118,5.472658e-18,412.617667,NaN,"(('bin_equal_frequency_5', 'yr_built'), ('IQR'...",N/A
74,EXP_45.9,6,3,61567.591960,12614.167407,0.001850,1.391619e-17,757.566667,NaN,"(('bin_equal_frequency_10', 'sqft_living'), ('...",N/A
61,EXP_45.10,6,3,61567.591960,12508.459862,0.000086,4.031199e-17,287.077333,NaN,"(('bin_equal_width_5', 'sqft_basement'), ('bin...",N/A
62,EXP_45.11,6,3,61567.591960,12508.459862,0.000086,4.031199e-17,377.317667,NaN,"(('bin_equal_width_5', 'sqft_basement'), ('bin...",N/A
63,EXP_45.12,4,3,61567.591960,12508.459862,0.000086,4.031199e-17,388.378000,NaN,"(('bin_equal_width_5', 'sqft_basement'), ('bin...",N/A
60,EXP_45.1,6,6,61567.591960,12086.625147,0.128299,2.257471e-16,191.312500,NaN,"(('bin_equal_frequency_5', 'yr_built'), ('bin_...",N/A
69,EXP_45.4,6,3,61567.591960,12194.232092,0.264906,2.257471e-16,98.576333,NaN,"(('bin_equal_frequency_5', 'yr_built'), ('bin_...",N/A


In [160]:
walmart_bin_shit = pd.read_csv("ate_bins_data_walmart.csv")

In [163]:
walmart_bin_shit

,bucket_range,min_ate_in_bin,max_ate_in_bin,min_length,best_sequence,count
0,"(25770.696, 25964.738]",25964.737538,25964.737538,3,"(('IQR', 'bedrooms'), ('bin_equal_frequency_10...",1
1,"(26352.82, 26546.862]",26427.764419,26427.764419,3,"(('zscore_clip_3', 'grade'), ('bin_equal_frequ...",1
2,"(26934.944, 27128.986]",26988.840916,26988.840916,3,"(('IQR', 'bedrooms'), ('bin_equal_frequency_10...",1
3,"(27323.027, 27517.069]",27417.658933,27417.658933,3,"(('IQR', 'bedrooms'), ('bin_equal_frequency_10...",1
4,"(28293.234, 28487.275]",28299.624570,28299.624570,3,"(('bin_equal_frequency_2', 'grade'), ('bin_equ...",1
...,...,...,...,...,...,...
283,"(83012.902, 83206.944]",83136.391138,83136.391138,3,"(('bin_equal_width_5', 'grade'), ('zscore_clip...",1
284,"(86311.606, 86505.647]",86346.308729,86346.308729,3,"(('bin_equal_width_2', 'floors'), ('bin_equal_...",1
285,"(86699.689, 86893.73]",86844.282495,86844.282495,3,"(('bin_equal_width_5', 'grade'), ('bin_equal_w...",1
286,"(91162.64, 91356.682]",91223.004178,91223.004178,3,"(('bin_equal_width_2', 'floors'), ('bin_equal_...",1


In [167]:
count_elements_relative_to_x(walmart_bin_shit, 10730)

In [164]:
import pandas as pd


def count_elements_relative_to_x(
    df: pd.DataFrame, x: float, direction: str = "smaller"
) -> int:
    """Counts elements smaller or bigger than x using bucket boundaries.

    Parameters:
    - df: pd.DataFrame with 'min_ate_in_bin', 'max_ate_in_bin', and 'count'
    columns.
    - x: The threshold value (float).
    - direction: 'smaller' to count elements < x, or 'bigger' to count elements
    > x.
    """
    if direction not in ["smaller", "bigger"]:
        raise ValueError("direction must be either 'smaller' or 'bigger'")

    if direction == "smaller":
        # Fully smaller: buckets where the max value is strictly less than x
        fully_covered = df[df["max_ate_in_bin"] < x]["count"].sum()

        # Partially smaller: bucket where x falls inside the min and max
        # boundaries
        partial_mask = (df["min_ate_in_bin"] <= x) & (
            df["max_ate_in_bin"] >= x
        )
        partial_match = df[partial_mask]

        if not partial_match.empty:
            row = partial_match.iloc[0]
            bin_range = row["max_ate_in_bin"] - row["min_ate_in_bin"]

            if bin_range > 0:
                # Linear interpolation to estimate the count below x
                proportion = (x - row["min_ate_in_bin"]) / bin_range
                fully_covered += round(proportion * row["count"])

        return int(fully_covered)

    else:  # direction == 'bigger'
        # Fully bigger: buckets where the min value is strictly greater than x
        fully_covered = df[df["min_ate_in_bin"] > x]["count"].sum()

        # Partially bigger: bucket where x falls inside the min and max
        # boundaries
        partial_mask = (df["min_ate_in_bin"] <= x) & (
            df["max_ate_in_bin"] >= x
        )
        partial_match = df[partial_mask]

        if not partial_match.empty:
            row = partial_match.iloc[0]
            bin_range = row["max_ate_in_bin"] - row["min_ate_in_bin"]

            if bin_range > 0:
                # Linear interpolation to estimate the count above x
                proportion = (row["max_ate_in_bin"] - x) / bin_range
                fully_covered += round(proportion * row["count"])

        return int(fully_covered)


In [2]:
x = pd.read_csv(
    "https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/datasets/TWINS/twin_pairs_X_3years_samesex.csv")

# The outcome data contains mortality of the lighter and heavier twin
y = pd.read_csv(
    "https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/datasets/TWINS/twin_pairs_Y_3years_samesex.csv")

# The treatment data contains weight in grams of both the twins
t = pd.read_csv(
    "https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/datasets/TWINS/twin_pairs_T_3years_samesex.csv")

In [3]:
# _0 denotes features specific to the lighter twin and _1 denotes features specific to the heavier twin
lighter_columns = ['pldel', 'birattnd', 'brstate', 'stoccfipb', 'mager8',
                   'ormoth', 'mrace', 'meduc6', 'dmar', 'mplbir', 'mpre5', 'adequacy',
                   'orfath', 'frace', 'birmon', 'gestat10', 'csex', 'anemia', 'cardiac',
                   'lung', 'diabetes', 'herpes', 'hydra', 'hemo', 'chyper', 'phyper',
                   'eclamp', 'incervix', 'pre4000', 'preterm', 'renal', 'rh', 'uterine',
                   'othermr', 'tobacco', 'alcohol', 'cigar6', 'drink5', 'crace',
                   'data_year', 'nprevistq', 'dfageq', 'feduc6', 'infant_id_0',
                   'dlivord_min', 'dtotord_min', 'bord_0',
                   'brstate_reg', 'stoccfipb_reg', 'mplbir_reg']
heavier_columns = ['pldel', 'birattnd', 'brstate', 'stoccfipb', 'mager8',
                   'ormoth', 'mrace', 'meduc6', 'dmar', 'mplbir', 'mpre5', 'adequacy',
                   'orfath', 'frace', 'birmon', 'gestat10', 'csex', 'anemia', 'cardiac',
                   'lung', 'diabetes', 'herpes', 'hydra', 'hemo', 'chyper', 'phyper',
                   'eclamp', 'incervix', 'pre4000', 'preterm', 'renal', 'rh', 'uterine',
                   'othermr', 'tobacco', 'alcohol', 'cigar6', 'drink5', 'crace',
                   'data_year', 'nprevistq', 'dfageq', 'feduc6',
                   'infant_id_1', 'dlivord_min', 'dtotord_min', 'bord_1',
                   'brstate_reg', 'stoccfipb_reg', 'mplbir_reg']

# Since data has pair property,processing the data to get separate row for each twin so that each child can be treated as an instance
data = []

for i in range(len(t.values)):

    # select only if both <=2kg
    if t.iloc[i].values[1] >= 2000 or t.iloc[i].values[2] >= 2000:
        continue
    this_instance_lighter = list(x.iloc[i][lighter_columns].values)
    this_instance_heavier = list(x.iloc[i][heavier_columns].values)

    # adding weight
    this_instance_lighter.append(t.iloc[i].values[1])
    this_instance_heavier.append(t.iloc[i].values[2])

    # adding treatment, is_heavier
    this_instance_lighter.append(0)
    this_instance_heavier.append(1)

    # adding the outcome
    this_instance_lighter.append(y.iloc[i].values[1])
    this_instance_heavier.append(y.iloc[i].values[2])
    data.append(this_instance_lighter)
    data.append(this_instance_heavier)

cols = ['pldel', 'birattnd', 'brstate', 'stoccfipb', 'mager8',
        'ormoth', 'mrace', 'meduc6', 'dmar', 'mplbir', 'mpre5', 'adequacy',
        'orfath', 'frace', 'birmon', 'gestat10', 'csex', 'anemia', 'cardiac',
        'lung', 'diabetes', 'herpes', 'hydra', 'hemo', 'chyper', 'phyper',
        'eclamp', 'incervix', 'pre4000', 'preterm', 'renal', 'rh', 'uterine',
        'othermr', 'tobacco', 'alcohol', 'cigar6', 'drink5', 'crace',
        'data_year', 'nprevistq', 'dfageq', 'feduc6',
        'infant_id', 'dlivord_min', 'dtotord_min', 'bord',
        'brstate_reg', 'stoccfipb_reg', 'mplbir_reg', 'wt', 'treatment', 'outcome']

df_twins_org = pd.DataFrame(columns=cols, data=data)
# df = df.drop(columns=['wt', 'infant_id', 'bord', 'dlivord_min', 'dtotord_min'])

In [4]:
df_twins_org_filled = df_twins_org.copy()
df_twins_org_filled.fillna(value=df_twins_org_filled.mean(),inplace=True)    #filling the missing values
df_twins_org_filled.fillna(value=df_twins_org_filled.mode().loc[0],inplace=True)

In [23]:
df_twins_org_filled['brstate_reg'].nunique()#describe()

In [7]:
treatment = 'treatment'
outcome = 'outcome'

# List your categorical columns (nominal 'u' and ordinal 'o')
# categorical_cols = [
#     'pldel', 'birattnd', 'ormoth', 'mrace',#, 'brstate', 'stoccfipb', 'mplbir'
#     'meduc6', 'mpre5', 'adequacy', 'orfath', 'frace', 'birmon',
#     'gestat10', 'cigar6', 'drink5', 'crace', 'feduc6', 'brstate_reg',
#     'stoccfipb_reg', 'mplbir_reg'
# ]
categorical_cols = ['brstate_reg', 'stoccfipb_reg', 'mplbir_reg', 'pldel' , 'mrace', 'frace', 'crace', 'orfath', 'ormoth','birmon']


df_encoded = df_twins_org_filled.copy()
df_encoded[categorical_cols] = df_encoded[categorical_cols].astype(str)

# 3. One-hot encode ONLY the categorical columns
# drop_first=True prevents the dummy variable trap in OLS regression
df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True)

# 4. Extract all the newly created dummy column names to feed to DoWhy
all_columns = df_encoded.columns.tolist()
# Your new confounders are everything except treatment, outcome, and non-confounders (like infant_id)
new_confounders = [col for col in all_columns if col not in [treatment, outcome, 'infant_id', 'data_year']]

In [9]:
df_encoded = df_encoded.astype(int)

In [10]:
# print(calculate_ate_linear_regression_lstsq(df_encoded, 'treatment', 'outcome', new_confounders))
print(calculate_ate_with_uncertainty(df_twins_org_filled, 'treatment', 'outcome', df_twins_org_filled.columns.difference(['treatment', 'outcome','infant_id', 'data_year'])))
print(calculate_ate_with_uncertainty(df_encoded, 'treatment', 'outcome', new_confounders))

{'ate': 0.06349380628894638, 'ci': (0.05578575700636361, 0.07120185557152915)}
{'ate': 0.06312781879647926, 'ci': (0.05543968414810987, 0.07081595344484866)}


In [11]:
print(calculate_ate_linear_regression_lstsq(df_encoded, 'treatment', 'outcome', new_confounders))

0.049650687440053444


In [39]:
print(calculate_ate_linear_regression_lstsq(df_twins_org_filled, 'treatment', 'outcome', df_twins_org_filled.columns.difference(['treatment', 'outcome','wt'])))

-0.007076226198019432


In [30]:
sequence_avi = (
    ('bin_equal_frequency_5', 'brstate'),
    ('bin_equal_frequency_5', 'stoccfipb'),
    ('bin_equal_frequency_5', 'mplbir'),
    ('bin_equal_frequency_5', 'infant_id'),
    ('bin_equal_frequency_5', 'wt'),
)
df_new = df_twins_org_filled.copy()
print(calculate_ate_linear_regression_lstsq(df_new, 'treatment', 'outcome', df_new.columns.difference(['treatment', 'outcome'])))
for func_name, col in sequence_avi:
    df_new = apply_data_preparations_seq(df_new,((func_name, col),), largest_data_transformations)
    print(calculate_ate_linear_regression_lstsq(df_new, 'treatment', 'outcome', df_new.columns.difference(['treatment', 'outcome'])))




0.061308184036284095
0.06341888484053217
0.06343057063952245
0.0634387696010261
0.06345112084161175
0.04874813783351515


In [21]:
def first_seen_qcut_col(series: pd.Series, q: int = 5) -> pd.Series:
    """
    Splits a Series into quantiles and labels them 1..q
    based on their order of first appearance.
    """
    # 1. Create the quantile intervals
    binned = pd.qcut(series, q=q, duplicates="drop")

    # 2. pd.factorize assigns 0, 1, 2... based on first appearance.
    # We add 1 to match the legacy 1-based indexing (1..5).
    first_seen_labels = pd.factorize(binned)[0] + 1

    # 3. Return as a new Series with the original index and name preserved
    return pd.Series(first_seen_labels, index=series.index, name=series.name)
sequence_avi = (
    ('bin_equal_frequency_5', 'brstate'),
    ('bin_equal_frequency_5', 'stoccfipb'),
    ('bin_equal_frequency_5', 'mplbir'),
    ('bin_equal_frequency_5', 'infant_id'),
    ('bin_equal_frequency_5', 'wt'),
)
df_new = df_twins_org_filled.copy()
print(calculate_ate_linear_regression_lstsq(df_new, 'treatment', 'outcome', df_new.columns.difference(['treatment', 'outcome'])))
for func_name, col in sequence_avi:
    df_new[col] = first_seen_qcut_col(df_new[col], q=5)
    print(calculate_ate_linear_regression_lstsq(df_new, 'treatment', 'outcome', df_new.columns.difference(['treatment', 'outcome'])))




0.06341930051246705
0.06343221579013239
0.06343727824696993
0.06343699788316069
0.06344800772194262
-0.01627111166641517


In [13]:
twins_avi_df = pd.read_csv('twins_avi.csv')


In [28]:
print(len(df_avi), len(twins_avi_df))
print(df_avi.isna().sum().sum(), twins_avi_df.isna().sum().sum())
print(df_avi.equals(twins_avi_df))
print(df_avi.fillna(df_avi.bfill()).equals(twins_avi_df))

23968 23968
98494 0
False
False


In [42]:
# for i in range(1):
#     # print(df_avi.iloc[i])
#     # print(twins_avi_df.iloc[i])
#     print(df_avi.sort_values(by='infant_id').iloc[i].compare(twins_avi_df.sort_values(by='infant_id').iloc[i]))
# twins_avi_df[twins_avi_df['infant_id']==1]


In [19]:
df_avi_filled = df_avi.fillna(df_avi.mean())

print(calculate_ate_linear_regression_lstsq(df_avi_filled, 'treatment', 'outcome', df_avi_filled.columns.difference(['treatment', 'outcome'])))

print(calculate_ate_linear_regression_lstsq(df_avi_filled, 'treatment', 'outcome', ['gestat10']))

print(calculate_ate_linear_regression_lstsq(twins_avi_df, 'treatment', 'outcome', twins_avi_df.columns.difference(['treatment', 'outcome'])))

print(calculate_ate_linear_regression_lstsq(df_avi, 'treatment', 'outcome', [
        'gestat10', 'wt', 'adequacy', 'nprevistq', 'hydra',
        'csex', 'dmar', 'data_year', 'dtotord_min', 'dlivord_min',
    ]))

im the mew alg!
0.06341930051246705
im the mew alg!
-0.02520026702269834
im the mew alg!
-0.01627111166642246
im the mew alg!


LinAlgError: SVD did not converge

In [14]:
#!/usr/bin/env python3
"""
Exact behavioral reconstruction of CURATE's datasets/twins/twins.csv
starting from the data-loading/preprocessing procedure shown in the
DoWhy v0.13 Twins example.

Sources:
  DoWhy notebook:
  https://www.pywhy.org/dowhy/v0.13/example_notebooks/dowhy_twins_example.html

  CURATE target CSV:
  https://raw.githubusercontent.com/avigailyam/CURATE-2026/refs/heads/main/datasets/twins/twins.csv

What this script reproduces:
  1. The DoWhy construction of one row per twin.
  2. The < 2000 g filter for both twins in a pair.
  3. DoWhy's missing-value imputation.
  4. The additional discretization recovered by direct comparison with
     CURATE's twins.csv.
  5. A strict cell-by-cell verification after CSV serialization/reload.

Important:
  The recovered discretization is behaviorally exact for the referenced
  CURATE CSV. This does not prove which original source-code function was
  used to create that CSV; it proves an equivalent transformation that
  reproduces it exactly.
"""

# from __future__ import annotations

import argparse
from pathlib import Path
from typing import Dict, Hashable, Tuple

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal


X_URL = (
    "https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/"
    "datasets/TWINS/twin_pairs_X_3years_samesex.csv"
)
Y_URL = (
    "https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/"
    "datasets/TWINS/twin_pairs_Y_3years_samesex.csv"
)
T_URL = (
    "https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/"
    "datasets/TWINS/twin_pairs_T_3years_samesex.csv"
)
CURATE_URL = (
    "https://raw.githubusercontent.com/avigailyam/CURATE-2026/"
    "refs/heads/main/datasets/twins/twins.csv"
)

LIGHTER_COLUMNS = [
    "pldel", "birattnd", "brstate", "stoccfipb", "mager8", "ormoth", "mrace",
    "meduc6", "dmar", "mplbir", "mpre5", "adequacy", "orfath", "frace",
    "birmon", "gestat10", "csex", "anemia", "cardiac", "lung", "diabetes",
    "herpes", "hydra", "hemo", "chyper", "phyper", "eclamp", "incervix",
    "pre4000", "preterm", "renal", "rh", "uterine", "othermr", "tobacco",
    "alcohol", "cigar6", "drink5", "crace", "data_year", "nprevistq",
    "dfageq", "feduc6", "infant_id_0", "dlivord_min", "dtotord_min", "bord_0",
    "brstate_reg", "stoccfipb_reg", "mplbir_reg",
]

HEAVIER_COLUMNS = [
    "pldel", "birattnd", "brstate", "stoccfipb", "mager8", "ormoth", "mrace",
    "meduc6", "dmar", "mplbir", "mpre5", "adequacy", "orfath", "frace",
    "birmon", "gestat10", "csex", "anemia", "cardiac", "lung", "diabetes",
    "herpes", "hydra", "hemo", "chyper", "phyper", "eclamp", "incervix",
    "pre4000", "preterm", "renal", "rh", "uterine", "othermr", "tobacco",
    "alcohol", "cigar6", "drink5", "crace", "data_year", "nprevistq",
    "dfageq", "feduc6", "infant_id_1", "dlivord_min", "dtotord_min", "bord_1",
    "brstate_reg", "stoccfipb_reg", "mplbir_reg",
]

OUTPUT_COLUMNS = [
    "pldel", "birattnd", "brstate", "stoccfipb", "mager8", "ormoth", "mrace",
    "meduc6", "dmar", "mplbir", "mpre5", "adequacy", "orfath", "frace",
    "birmon", "gestat10", "csex", "anemia", "cardiac", "lung", "diabetes",
    "herpes", "hydra", "hemo", "chyper", "phyper", "eclamp", "incervix",
    "pre4000", "preterm", "renal", "rh", "uterine", "othermr", "tobacco",
    "alcohol", "cigar6", "drink5", "crace", "data_year", "nprevistq",
    "dfageq", "feduc6", "infant_id", "dlivord_min", "dtotord_min", "bord",
    "brstate_reg", "stoccfipb_reg", "mplbir_reg", "wt", "treatment", "outcome",
]

EXPECTED_DISCRETIZED = ["brstate", "stoccfipb", "mplbir", "infant_id", "wt"]


def read_csv(source: str) -> pd.DataFrame:
    """Read either a local CSV path or an HTTP(S) URL."""
    return pd.read_csv(source)


def build_dowhy_dataframe(
    x: pd.DataFrame,
    y: pd.DataFrame,
    t: pd.DataFrame,
) -> Tuple[pd.DataFrame, int]:
    """Reproduce the construction and imputation shown in DoWhy v0.13."""
    rows = []

    for i in range(len(t)):
        # DoWhy keeps a pair only when BOTH twins weigh < 2000 g.
        if t.iloc[i].values[1] >= 2000 or t.iloc[i].values[2] >= 2000:
            continue

        lighter = list(x.iloc[i][LIGHTER_COLUMNS].values)
        heavier = list(x.iloc[i][HEAVIER_COLUMNS].values)

        # Birth weight.
        lighter.append(t.iloc[i].values[1])
        heavier.append(t.iloc[i].values[2])

        # Treatment: lighter twin = 0, heavier twin = 1.
        lighter.append(0)
        heavier.append(1)

        # Mortality outcome.
        lighter.append(y.iloc[i].values[1])
        heavier.append(y.iloc[i].values[2])

        rows.append(lighter)
        rows.append(heavier)

    df = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)
    missing_before = int(df.isna().sum().sum())

    # These are the two imputation lines used in the DoWhy notebook.
    df = df.astype({"treatment": "bool"}, copy=False)
    df.fillna(value=df.mean(), inplace=True)
    df.fillna(value=df.mode().loc[0], inplace=True)

    return df, missing_before


def first_seen_qcut(
    series: pd.Series,
    q: int = 5,
) -> Tuple[pd.Series, Dict[Hashable, int]]:
    """
    Split into q quantile bins, then encode the bins as 1..q in the order
    in which each bin first appears in the dataset.
    """
    binned = pd.qcut(series, q=q, duplicates="drop")

    mapping: Dict[Hashable, int] = {}
    encoded = []
    next_label = 1

    for interval in binned:
        if interval not in mapping:
            mapping[interval] = next_label
            next_label += 1
        encoded.append(mapping[interval])

    return pd.Series(encoded, index=series.index, dtype="int64"), mapping


def apply_recovered_discretization(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, list[str], dict[str, Dict[Hashable, int]]]:
    """
    Exact behavioral rule recovered from comparison with CURATE's CSV:

      - Find columns with > 50 distinct values.
      - Discretize each into 5 quantile bins.
      - Label the bins 1..5 by first appearance, not by numeric interval order.
    """
    out = df.copy()
    discretized = [
        col for col in out.columns
        if out[col].nunique(dropna=False) > 50
    ]

    mappings: dict[str, Dict[Hashable, int]] = {}
    for col in discretized:
        out[col], mappings[col] = first_seen_qcut(out[col], q=5)

    # CURATE stores treatment as 0/1 in the CSV.
    out["treatment"] = out["treatment"].astype(int)

    return out, discretized, mappings


def in_memory_exact_counts(
    reconstructed: pd.DataFrame,
    target: pd.DataFrame,
) -> Tuple[int, int, float]:
    """Count strict float equality before writing the reconstructed CSV."""
    a = reconstructed.to_numpy(dtype=float)
    b = target.to_numpy(dtype=float)

    exact = (a == b)
    equal_cells = int(exact.sum())
    different_cells = int((~exact).sum())
    max_abs_diff = float(np.max(np.abs(a - b)))

    return equal_cells, different_cells, max_abs_diff


def verify_exact_csv_roundtrip(
    reconstructed: pd.DataFrame,
    target: pd.DataFrame,
    output_path: str,
) -> Tuple[int, int]:
    """
    Save the reconstructed dataframe exactly as a CSV, read it back, and
    compare it with CURATE's target CSV using strict equality.
    """
    reconstructed.to_csv(output_path, index=False)
    reloaded = pd.read_csv(output_path)

    if reloaded.shape != target.shape:
        raise AssertionError(
            f"Shape mismatch after reload: {reloaded.shape} vs {target.shape}"
        )

    if list(reloaded.columns) != list(target.columns):
        raise AssertionError("Column names/order do not match the target CSV.")

    # Strict pandas check: exact values and inferred dtypes must match.
    assert_frame_equal(reloaded, target, check_exact=True)

    equal_cells = int((reloaded.to_numpy() == target.to_numpy()).sum())
    total_cells = int(target.shape[0] * target.shape[1])
    different_cells = total_cells - equal_cells

    return equal_cells, different_cells


def main() -> None:
    # parser = argparse.ArgumentParser(
    #     description="Reconstruct CURATE twins.csv from the DoWhy v0.13 Twins preprocessing."
    # )
    # parser.add_argument("--x", default=X_URL, help="CEVAE covariates CSV URL/path")
    # parser.add_argument("--y", default=Y_URL, help="CEVAE outcomes CSV URL/path")
    # parser.add_argument("--t", default=T_URL, help="CEVAE weights CSV URL/path")
    # parser.add_argument("--target", default=CURATE_URL, help="CURATE twins.csv URL/path")
    # parser.add_argument(
    #     "--output",
    #     default="twins_reconstructed.csv",
    #     help="Output path for reconstructed CSV",
    # )
    # args = parser.parse_args()
    #
    # x = read_csv(args.x)
    # y = read_csv(args.y)
    # t = read_csv(args.t)
    # target = read_csv(args.target)

    dowhy_df, missing_before = build_dowhy_dataframe(X_URL, Y_URL, T_URL)
    reconstructed, discretized, mappings = apply_recovered_discretization(dowhy_df)

    # Structural checks established during reconstruction.
    assert dowhy_df.shape == (23968, 53), dowhy_df.shape
    assert missing_before == 98494, missing_before
    assert int(dowhy_df.isna().sum().sum()) == 0
    assert discretized == EXPECTED_DISCRETIZED, discretized
    assert reconstructed.shape == target.shape
    assert list(reconstructed.columns) == list(target.columns)

    total_cells = int(target.shape[0] * target.shape[1])

    mem_equal, mem_different, max_abs_diff = in_memory_exact_counts(
        reconstructed, target
    )

    csv_equal, csv_different = verify_exact_csv_roundtrip(
        reconstructed, target, args.output
    )

    print("PASS")
    print(f"Shape: {target.shape[0]:,} rows x {target.shape[1]} columns")
    print(f"Total cells: {total_cells:,}")
    print(f"Missing values before DoWhy imputation: {missing_before:,}")
    print(
        "Missing values after DoWhy imputation: "
        f"{int(dowhy_df.isna().sum().sum()):,}"
    )
    print("Discretized columns:", ", ".join(discretized))
    print()
    print("Before CSV serialization:")
    print(f"  Strictly equal cells: {mem_equal:,}")
    print(f"  Strictly different cells: {mem_different:,}")
    print(f"  Maximum absolute difference: {max_abs_diff:.3e}")
    print("  The differences are only floating-point representation effects.")
    print()
    print("After saving the reconstruction to CSV and reading it back:")
    print(f"  Strictly equal cells: {csv_equal:,} / {total_cells:,}")
    print(f"  Strictly different cells: {csv_different:,}")
    print("  Exact dataframe equality (including inferred dtypes): PASS")
    print()
    print(f"Saved reconstructed CSV: {args.output}")
    print()
    print("Recovered interval -> encoded-label mappings:")
    for col in discretized:
        print(f"  {col}:")
        for interval, label in mappings[col].items():
            print(f"    {interval} -> {label}")


if __name__ == "__main__":
    main()

AttributeError: 'str' object has no attribute 'iloc'